# Data Loading Demonstration


In [ ]:
%pip install pandas pyarrow tqdm

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import sys
sys.path.append('./lib')

## Loading Triage Malware Dataset

The dataset contains annotated TLS communication samples collected from a Triage-based malware sandbox environment.  
Each sample corresponds to network traffic generated during the execution of a malware specimen under controlled conditions.

The following columns represent the relevant semantic annotations used for analysis and modeling:

- **meta.application** — Application identifier. This field is not used in this dataset and is therefore set to `NA`.
- **meta.system** — Operating system metadata associated with the communication, typically indicating the OS family and version of the sandboxed environment.
- **meta.service** — Network-facing service or operating system subsystem derived from system metadata, enabling coarse-grained service-level classification.
- **meta.malware.family** — Malware family annotation. Samples without a confidently identified family are labeled as `NA`.
- **meta.host** — Host-level identifier or role. This field is not used in this dataset and is therefore set to `NA`.

Note that a single network connection may be annotated both with a malware family and a system service.  
In such cases, the connection is *triggered by the execution of the malware sample*, but the observed TLS communication corresponds to a **legitimate operating system service** (e.g., update checks, certificate validation, or telemetry) invoked by the malware rather than to a malware-specific network protocol.


## Load all data

In [2]:
from data_helper import get_families_frequency
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_malware = ds.dataset(f"../datasets/malware.parquet", format="parquet")


df_malware = ds_malware.to_table().to_pandas()
get_families_frequency(df_malware)

,count,count
0,44caliber,232
1,ades_stealer,19526
2,agenttesla,3006
3,amadey,45718
4,ardamax,70
...,...,...
110,xmrig_linux,34
111,xorist,19526
112,xred,615
113,xtremerat,1


## Loading per family
The dataset allows you to load only the columns you need, so you can collect information on available families without loading the entire dataset.
Then, the individual family datasets are loaded.

In [3]:
# Collect available familes:
families = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .column("meta.malware.family")
    .unique()
    .to_pylist()
)
print(families)
# Load familes one by one:
for family in families:
    if family is not None:
        df_family = ds_malware.to_table(filter=(pc.field("meta.malware.family") == family)).to_pandas()
        print(f"{family} - {df_family.shape[0]} / {len(df_family)}")

[None, 'agenttesla', 'asyncrat', 'stormkitty', 'blackmoon', 'blihanstealer', 'cobaltstrike', 'floxif', 'gh0strat', 'lockbit', 'lumma', 'neconyd', 'purplefox', 'quasar', 'ramnit', 'redline', 'sectoprat', 'smokeloader', 'stealc', 'umbral', 'vidar', 'xmrig', 'xred', 'xworm', 'cosmu', 'gink', 'sparkrat', 'vipkeylogger', 'discordrat', 'blankgrabber', 'cryptolocker', 'formbook', 'sliver', 'guloader', 'remcos', 'masslogger', 'matiex', 'snakekeylogger', 'modiloader', 'amadey', 'koiloader', 'mimikatz', 'njrat', 'aurotun', 'wannacry', 'salatstealer', 'deerstealer', 'donutloader', 'socks5systemz', 'venomrat', 'rhadamanthys', 'neshta', 'nightingale', 'azorult', 'lokibot', 'rms', 'xenorat', 'metasploit', 'nanocore', 'gcleaner', 'hijackloader', 'berbew', 'warzonerat', 'revengerat', 'cyber_stealer', 'babylonrat', 'emotet', 'raccoon', 'ades_stealer', 'chaos', 'dcrat', 'dragonforce', 'jlocker', 'onlylogger', 'rokrat', 'xorist', 'trickbot', 'coffloader', 'raworld', 'sality', 'darkvision', 'xmrig_linux',

It is possible to do more data analysis without reading the entire dataset using dataset. For instance, we can compute the count of samples for every malware family.

In [4]:
freq = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .to_pandas()
    ["meta.malware.family"]
    .value_counts()
    .sort_index()
)
freq

meta.malware.family
44caliber         232
ades_stealer    19526
agenttesla       3006
amadey          45718
ardamax            70
                ...  
xmrig_linux        34
xorist          19526
xred              615
xtremerat           1
xworm           44085
Name: count, Length: 115, dtype: int64

It is often best to load only the necessary columns and then filter or compute on the data frame.

In [6]:
_df = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .to_pandas())
non_malware_samples = _df["meta.malware.family"].isna().sum()
print(f"Non malware samples={non_malware_samples}")

Non malware samples=576908


## Loading Windows Applications Dataset

The dataset contains annotated TLS communication samples collected from applications executed in a controlled Windows environment.  
Traffic was generated by legitimate Windows-based applications running inside **Windows Sandbox**, ensuring a clean and reproducible execution context.

The following columns represent the relevant semantic annotations used for analysis and modeling:

- **meta.application** — Application identifier in the normalized form `VENDOR.APPLICATION`, representing the originating software (e.g., `MICROSOFT.OUTLOOK`, `GOOGLE.CHROME`).
- **meta.system** — Operating system metadata associated with the communication, indicating the OS family and version of the sandboxed environment.
- **meta.service** — Network-facing service annotation. This field is not used in this dataset and is set to `NA`.
- **meta.malware** — Malware family annotation. This field is not used in this dataset and is set to `NA`.
- **meta.host** — Host-level identifier or role. This field is not used in this dataset and is set to `NA`.

The dataset includes only TLS connections directly attributable to the executed applications.  
Background operating system traffic and unrelated connections were explicitly filtered out to preserve a clean, application-centric view of network behavior.

In [9]:
from data_helper import get_applications_frequency
import pyarrow.dataset as ds
import pyarrow.compute as pc

dataset = ds.dataset(f"../datasets/winapps.parquet", format="parquet")
print("Loading dataset...")
df_applications = dataset.to_table().to_pandas()

get_applications_frequency(df_applications)

Loading dataset...


,count,count
0,Adamant.Messenger,2246
1,AirDroid.AirDroid,3352
2,Apple.AppleApplicationSupport.x86,33
3,Apple.iTunes,32
4,Asana.Asana,463
...,...,...
59,agalwood.Motrix,32
60,deltachat.deltachat,32
61,eMClient.eMClient,169
62,pCloudAG.pCloudDrive,171
